# D&D Beyond Species Sync

Sync homebrew species from Google Sheets to D&D Beyond.

## Setup Instructions

1. Get your D&D Beyond cookies (Copy as cURL and extract Cookie header).
2. Get security tokens from the create species page.
3. Update your `.env` with `DDB_COOKIES`, `DDB_SECURITY_TOKEN`, and `DDB_AUTHENTICITY_TOKEN`.


## 1. Setup and Imports

In [1]:
import pandas as pd
import requests
import json
import time
import re
import os
from typing import Dict, List, Optional
from dotenv import load_dotenv
from datetime import datetime
from pathlib import Path
import sys

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from Spreadsheet.sheets import ContentSheetsClient
from FiveETools.core.fantasy.species import species_list

fantasy_sheets = ContentSheetsClient("fantasy", credentials_path=str(repo_root / "FiveETools" / "key.json"))

from DNDBeyond.core.Helpers.DnDBeyondAPI import DnDBeyondAPI
from DNDBeyond.core.Helpers import convert_species_to_ddb, normalize_ddb_id
from DNDBeyond.core.Helpers.converter import refresh_species_field_ids_from_html_text

print("✓ Imports loaded")


✓ Imports loaded


## 2. D&D Beyond Authentication

In [2]:
# ============================================
# D&D Beyond Configuration
# ============================================

load_dotenv()

DDB_BASE_URL = os.getenv("DDB_BASE_URL") or "https://www.dndbeyond.com"
DDB_COOKIES = os.getenv("DDB_COOKIES")
DDB_SECURITY_TOKEN = os.getenv("DDB_SECURITY_TOKEN")
DDB_AUTHENTICITY_TOKEN = os.getenv("DDB_AUTHENTICITY_TOKEN")
REQUEST_VERIFICATION_TOKEN = os.getenv("REQUEST_VERIFICATION_TOKEN")
DDB_USER_ID = os.getenv("DDB_USER_ID")
DDB_USERNAME = os.getenv("DDB_USERNAME")

# ============================================
# Setup Session
# ============================================

session = requests.Session()

if DDB_COOKIES:
    session.headers.update({
        "Cookie": DDB_COOKIES,
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br, zstd",
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:147.0) Gecko/20100101 Firefox/147.0",
        "Referer": f"{DDB_BASE_URL}/homebrew/creations/create-species/create",
        "Origin": DDB_BASE_URL,
        "Sec-Fetch-Dest": "document",
        "Sec-Fetch-Mode": "navigate",
        "Sec-Fetch-Site": "same-origin",
        "Sec-Fetch-User": "?1",
        "Upgrade-Insecure-Requests": "1"
    })

    try:
        response = session.get(f"{DDB_BASE_URL}/homebrew/creations/create-species/create", timeout=30)
        refresh_species_field_ids_from_html_text(response.text)
        print("✓ Refreshed species field IDs from live page")
    except Exception as exc:
        print(f"⚠️  Could not refresh species field IDs: {exc}")

    print("✓ Using cookie authentication")
    print(f"✓ Base URL: {DDB_BASE_URL}")
    print(f"✓ User: {DDB_USERNAME} (ID: {DDB_USER_ID})")

    if DDB_SECURITY_TOKEN and DDB_AUTHENTICITY_TOKEN:
        print("✓ Security tokens configured")
    else:
        print("⚠️  Security tokens not set - you'll need these to create species!")
else:
    print("✗ ERROR: No cookies provided!")
    print("   Please set DDB_COOKIES with your browser session cookies")


✓ Refreshed species field IDs from live page
✓ Using cookie authentication
✓ Base URL: https://www.dndbeyond.com
✓ User: Rockoteque (ID: 110746825)
✓ Security tokens configured


## 3. Load Species Data

In [3]:
print("Loading species from Google Sheets...")
print(f"✓ Loaded {len(species_list)} species")

if species_list:
    sample = species_list[0]
    print(f"Sample species: {sample.get('name')}")


Loading species from Google Sheets...
✓ Loaded 58 species
Sample species: Felkin


## 4. D&D Beyond API Helper

In [4]:
ddb_api = DnDBeyondAPI(
    session,
    DDB_SECURITY_TOKEN,
    DDB_AUTHENTICITY_TOKEN,
    verification_token=REQUEST_VERIFICATION_TOKEN,
)
print("✓ D&D Beyond API initialized")


✓ D&D Beyond API initialized


In [5]:
# ============================================
# Species helpers
# ============================================

from DNDBeyond.core.Helpers import get_species_field_ids

ASSET_ROOT = None
if 'repo_root' in globals():
    ASSET_ROOT = repo_root / "assets" / "art" / "species" / "fantasy"
else:
    ASSET_ROOT = Path("assets/art/species/fantasy")

def _asset_key(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9]+", "", value or "").lower()

def _build_species_asset_map():
    assets = {}
    if not ASSET_ROOT.exists():
        return assets
    for path in ASSET_ROOT.glob("*.png"):
        stem = path.stem
        suffix = None
        base = stem
        if stem.endswith("_F"):
            suffix = "F"
            base = stem[:-2]
        elif stem.endswith("_M"):
            suffix = "M"
            base = stem[:-2]
        key = _asset_key(base)
        if not key:
            continue
        assets.setdefault(key, {})
        if suffix:
            assets[key][suffix] = path
        else:
            assets[key].setdefault("any", path)
    return assets

SPECIES_ASSETS = _build_species_asset_map()
DEFAULT_SPECIES_AVATAR = None
if ASSET_ROOT.exists():
    pngs = sorted(ASSET_ROOT.glob("*.png"))
    if pngs:
        DEFAULT_SPECIES_AVATAR = pngs[0]

def _resolve_species_avatar_paths(species_data: dict):
    small_path = species_data.get("avatar_small_path")
    large_path = species_data.get("avatar_large_path")

    if small_path and large_path:
        return small_path, large_path

    name_key = _asset_key(species_data.get("name", ""))
    asset_entry = SPECIES_ASSETS.get(name_key)
    if not asset_entry:
        aliases = species_data.get("alias") or []
        if isinstance(aliases, str):
            aliases = [aliases]
        for alias in aliases:
            alias_key = _asset_key(alias)
            if alias_key in SPECIES_ASSETS:
                asset_entry = SPECIES_ASSETS[alias_key]
                break
    if not asset_entry:
        asset_entry = {}

    if not small_path:
        small_path = asset_entry.get("F") or asset_entry.get("M") or asset_entry.get("any")
    if not large_path:
        large_path = asset_entry.get("M") or asset_entry.get("F") or asset_entry.get("any") or small_path

    if not small_path and DEFAULT_SPECIES_AVATAR:
        small_path = DEFAULT_SPECIES_AVATAR
    if not large_path and DEFAULT_SPECIES_AVATAR:
        large_path = DEFAULT_SPECIES_AVATAR

    if small_path:
        species_data["avatar_small_path"] = str(small_path)
    if large_path:
        species_data["avatar_large_path"] = str(large_path)

    return small_path, large_path

def _open_species_files(species_data: dict, required_file_fields: Optional[set] = None):
    field_ids = get_species_field_ids()
    files = {}
    _resolve_species_avatar_paths(species_data)
    small_path = species_data.get("avatar_small_path")
    large_path = species_data.get("avatar_large_path")
    if small_path and field_ids.get("avatar_small"):
        path = Path(small_path)
        if path.exists():
            files[field_ids["avatar_small"]] = open(path, "rb")
    if large_path and field_ids.get("avatar_large"):
        path = Path(large_path)
        if path.exists():
            files[field_ids["avatar_large"]] = open(path, "rb")

    if required_file_fields:
        missing = [field for field in required_file_fields if field not in files]
        if missing:
            fallback = small_path or large_path or DEFAULT_SPECIES_AVATAR
            if fallback:
                fallback_path = Path(fallback)
                if fallback_path.exists():
                    for field in missing:
                        files[field] = open(fallback_path, "rb")

    return files

def _close_species_files(files: dict):
    for f in files.values():
        try:
            f.close()
        except Exception:
            pass

def _validate_species_form(form_data: dict, required_fields: set, required_files: set, files: dict):
    missing_fields = [
        key for key in required_fields
        if key not in form_data or str(form_data.get(key, "")).strip() == ""
    ]
    missing_files = [key for key in required_files if key not in files]
    return {"missing_fields": missing_fields, "missing_files": missing_files}

print("✓ Species helpers ready")


✓ Species helpers ready


## 5. Sync Species to D&D Beyond

In [6]:
DRY_RUN = False
BATCH_SIZE = 5
DELAY = 1
VERBOSE = True
SKIP_EXISTING = True
UPDATE_EXISTING = True

print("=" * 60)
if DRY_RUN:
    print("DRY RUN - No species will be created/updated")
else:
    print("⚠️  LIVE MODE - Species WILL be created/updated!")
print("=" * 60)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Batch size: {'ALL SPECIES' if BATCH_SIZE is None else BATCH_SIZE}")
print(f"Delay between requests: {DELAY}s")
print(f"Verbose logging: {'ON' if VERBOSE else 'OFF'}")
print(f"Skip existing: {'YES' if SKIP_EXISTING else 'NO (will attempt to create anyway)'}")
print(f"Update existing: {'YES (will update when DDB ID exists)' if UPDATE_EXISTING else 'NO'}")
print()

species_to_sync = species_list if BATCH_SIZE is None else species_list[:BATCH_SIZE]
print(f"Total species to process: {len(species_to_sync)}")
print()

print("Checking spreadsheet for existing DDB IDs...")
df_species = fantasy_sheets.get_sheet("993815941")
SPECIES_NAME_COLUMN = "Name"
species_to_ddb_id = {}
if 'DDB' in df_species.columns and SPECIES_NAME_COLUMN in df_species.columns:
    for _, row in df_species.iterrows():
        species_name = row.get(SPECIES_NAME_COLUMN)
        ddb_id = row.get('DDB')
        if species_name and pd.notna(ddb_id):
            normalized_id = normalize_ddb_id(ddb_id)
            if normalized_id:
                species_to_ddb_id[species_name] = normalized_id
    print(f"✓ Found {len(species_to_ddb_id)} species with DDB IDs in spreadsheet")
else:
    print("  No DDB IDs found in spreadsheet")

print("\nFetching existing homebrew species from D&D Beyond...")
if not DRY_RUN:
    try:
        existing_species = ddb_api.species.list()
        print(f"✓ Found {len(existing_species)} existing homebrew species\n")
    except Exception as exc:
        print(f"⚠️  Could not fetch existing species: {exc}")
        existing_species = []

REQUIRED_FORM_FIELDS = set()
REQUIRED_FILE_FIELDS = set()
if not DRY_RUN:
    try:
        response = session.get(f"{DDB_BASE_URL}/homebrew/creations/create-species/create", timeout=30)
        refresh_species_field_ids_from_html_text(response.text)
        REQUIRED_FORM_FIELDS = set(re.findall(r'name="([^"]+)"[^>]*data-validation-required="true"', response.text))
        required_patterns = [
            r'req[^>]*>Small Avatar</div>.*?<input[^>]+type="file"[^>]+name="([^"]+)"',
            r'req[^>]*>Large Avatar</div>.*?<input[^>]+type="file"[^>]+name="([^"]+)"',
        ]
        for pattern in required_patterns:
            match = re.search(pattern, response.text, re.S)
            if match:
                REQUIRED_FILE_FIELDS.add(match.group(1))
        print("✓ Refreshed required species fields from live page")
    except Exception as exc:
        print(f"⚠️  Could not refresh required species fields: {exc}")
else:
    existing_species = []

results = {
    'created': 0,
    'updated': 0,
    'skipped': 0,
    'errors': 0,
    'details': []
}

spreadsheet_updates = []

for i, species in enumerate(species_to_sync, 1):
    species_name = species.get('name', 'Unnamed')
    print(f"[{i}/{len(species_to_sync)}] Processing: {species_name}")

    try:
        if VERBOSE:
            print("  → Converting species to D&D Beyond format...")
        try:
            ddb_form = convert_species_to_ddb(species)
            if VERBOSE:
                print(f"    Size: {ddb_form.get('size')} Speed: {ddb_form.get('speed-walking')}")
        except Exception as conv_error:
            print(f"  ✗ Conversion failed: {conv_error}")
            results['errors'] += 1
            results['details'].append({
                'title': species_name,
                'action': 'error',
                'success': False,
                'error': f"Conversion error: {str(conv_error)}",
                'error_type': 'conversion'
            })
            continue

        existing_id = species_to_ddb_id.get(species_name)
        if existing_id:
            if UPDATE_EXISTING and not SKIP_EXISTING:
                if DRY_RUN:
                    print(f"  → DRY RUN: Would update species (ID: {existing_id})")
                    results['updated'] += 1
                    results['details'].append({
                        'title': species_name,
                        'action': 'dry_run_update',
                        'success': True,
                        'id': existing_id
                    })
                else:
                    print(f"  → Updating existing species (ID: {existing_id})")
                    slug = DnDBeyondAPI.create_slug(species_name)
                    updated = ddb_api.species.update(existing_id, slug, {'form_data': ddb_form})
                    if updated:
                        print("  ✓ Updated species successfully")
                        results['updated'] += 1
                        results['details'].append({
                            'title': species_name,
                            'action': 'updated',
                            'success': True,
                            'id': existing_id
                        })
                    else:
                        print("  ✗ Failed to update species")
                        results['errors'] += 1
                        results['details'].append({
                            'title': species_name,
                            'action': 'error',
                            'success': False,
                            'error_type': 'update_failed',
                            'id': existing_id,
                            'error': ddb_api.last_error
                        })
                if i < len(species_to_sync):
                    time.sleep(DELAY)
            else:
                print(f"  ✓ Already synced (DDB ID: {existing_id})")
                results['skipped'] += 1
                results['details'].append({
                    'title': species_name,
                    'action': 'skipped',
                    'success': True,
                    'id': existing_id,
                    'reason': 'already_in_spreadsheet'
                })
            continue

        if SKIP_EXISTING and not DRY_RUN and existing_species:
            existing_id = ddb_api.species.find_by_name(species_name, existing_species)
            if existing_id:
                print(f"  ⚠️  Species exists on D&D Beyond (ID: {existing_id})")
                spreadsheet_updates.append({
                    'match_value': species_name,
                    'update_column': 'DDB',
                    'update_value': existing_id
                })
                results['skipped'] += 1
                results['details'].append({
                    'title': species_name,
                    'action': 'skipped',
                    'success': True,
                    'id': existing_id,
                    'reason': 'already_exists_ddb'
                })
                continue

        if DRY_RUN:
            print("  → DRY RUN: Would create species")
            results['created'] += 1
            results['details'].append({
                'title': species_name,
                'action': 'dry_run_create',
                'success': True
            })
        else:
            print("  → Creating species on D&D Beyond...")
            files = _open_species_files(species, REQUIRED_FILE_FIELDS)
            validation = _validate_species_form(ddb_form, REQUIRED_FORM_FIELDS, REQUIRED_FILE_FIELDS, files)
            if validation["missing_fields"] or validation["missing_files"]:
                print("  ✗ Missing required fields/files for create")
                if validation["missing_fields"]:
                    print(f"    Missing fields: {validation['missing_fields']}")
                if validation["missing_files"]:
                    print(f"    Missing files: {validation['missing_files']}")
                _close_species_files(files)
                results['errors'] += 1
                results['details'].append({
                    'title': species_name,
                    'action': 'error',
                    'success': False,
                    'error_type': 'validation_failed',
                    'missing_fields': validation['missing_fields'],
                    'missing_files': validation['missing_files'],
                })
                continue
            species_id = ddb_api.species.create({'form_data': ddb_form, 'files': files})
            _close_species_files(files)
            if species_id:
                print(f"  ✓ Created: ID {species_id}")
                spreadsheet_updates.append({
                    'match_value': species_name,
                    'update_column': 'DDB',
                    'update_value': species_id
                })
                results['created'] += 1
                results['details'].append({
                    'title': species_name,
                    'action': 'created',
                    'success': True,
                    'id': species_id
                })
                existing_species.append({'name': species_name, 'id': species_id})
            else:
                print("  ✗ Failed to create species")
                results['errors'] += 1
                results['details'].append({
                    'title': species_name,
                    'action': 'error',
                    'success': False,
                    'error_type': 'creation_failed',
                    'error': ddb_api.last_error
                })
            if i < len(species_to_sync):
                time.sleep(DELAY)

    except Exception as exc:
        print(f"  ✗ Unexpected error: {exc}")
        results['errors'] += 1
        results['details'].append({
            'title': species_name,
            'action': 'error',
            'success': False,
            'error_type': 'unexpected_exception',
            'error': str(exc)
        })

# Write DDB IDs back to spreadsheet
if spreadsheet_updates and not DRY_RUN:
    print('\n' + '=' * 60)
    print('Updating Google Sheets with DDB IDs...')
    print('=' * 60)
    try:
        update_results = fantasy_sheets.batch_update_cells_by_row_match(
            "993815941",
            SPECIES_NAME_COLUMN,
            spreadsheet_updates
        )
        success_count = sum(1 for v in update_results.values() if v)
        print(f"✓ Updated {success_count}/{len(spreadsheet_updates)} species in spreadsheet")
    except Exception as exc:
        print(f"✗ Error updating spreadsheet: {exc}")

print()
print('=' * 60)
print('SYNC COMPLETE')
print('=' * 60)
if DRY_RUN:
    print(f"DRY RUN: Would create {results['created']} species")
    print(f"DRY RUN: Would update {results['updated']} species")
else:
    print(f"✓ Created: {results['created']}")
    print(f"✓ Updated: {results['updated']}")
    print(f"⚠️  Skipped (already exist): {results['skipped']}")
    print(f"✗ Errors: {results['errors']}")

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
log_file = f"out/species_sync_log_{timestamp}.json"
log_dir = os.path.dirname(log_file)
os.makedirs(log_dir, exist_ok=True)
with open(log_file, 'w') as f:
    json.dump({
        'timestamp': timestamp,
        'dry_run': DRY_RUN,
        'verbose': VERBOSE,
        'batch_size': BATCH_SIZE,
        'skip_existing': SKIP_EXISTING,
        'update_existing': UPDATE_EXISTING,
        'total': len(species_to_sync),
        'created': results['created'],
        'updated': results['updated'],
        'skipped': results['skipped'],
        'errors': results['errors'],
        'details': results['details']
    }, f, indent=2)

print(f"\n✓ Log saved to: {log_file}")


⚠️  LIVE MODE - Species WILL be created/updated!
Start time: 2025-12-26 09:51:21
Batch size: 5
Delay between requests: 1s
Verbose logging: ON
Skip existing: YES
Update existing: YES (will update when DDB ID exists)

Total species to process: 5

Checking spreadsheet for existing DDB IDs...
✓ Found 1 species with DDB IDs in spreadsheet

Fetching existing homebrew species from D&D Beyond...
✓ Found 0 existing homebrew species

✓ Refreshed required species fields from live page
[1/5] Processing: Felkin
  → Converting species to D&D Beyond format...
    Size: 4 Speed: 35
  → Creating species on D&D Beyond...
  ✗ Missing required fields/files for create
    Missing files: ['fba27e0f907869c188a4876c6116e2416']
[2/5] Processing: Canid
  → Converting species to D&D Beyond format...
    Size: 4 Speed: 40
  → Creating species on D&D Beyond...
  ✗ Missing required fields/files for create
    Missing files: ['fba27e0f907869c188a4876c6116e2416']
[3/5] Processing: Rascan
  → Converting species to D&D